# Setup: Generate Sample Dataset

This cell creates the required folder structure (`data/raw/` and `data/processed/`) relative to the notebook, and generates the sample CSV dataset with missing values. 
This ensures the dataset is ready for cleaning functions and saves it to `data/raw/sample_data.csv`.

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    ("src/cleaning.py", "NEEDED", "YOU write this in the homework - the import fails until you do"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: c:\Users\hunyuan\bootcamp_yuan_hun\homework\homework06

  [OK ]  NEEDED    src/cleaning.py                     YOU write this in the homework - the import fails until you do

All needed files present.


In [3]:
import os
import pandas as pd
import numpy as np

# Define folder paths relative to this notebook
raw_dir = 'data/raw'
processed_dir = 'data/processed'

# Create folders if they don't exist
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

# Define the sample data
data = {
    'age': [34, 45, 29, 50, 38, np.nan, 41],
    'income': [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    'score': [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    'zipcode': ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
    'city': ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco'],
    'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan]
}

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV in raw data folder
csv_path = os.path.join(raw_dir, 'sample_data.csv')
if not os.path.exists(csv_path):
    df.to_csv(csv_path, index=False)
    print(f'Sample dataset created and saved to {csv_path}')
else:
    print(f'File already exists at {csv_path}. Skipping CSV creation to avoid overwrite.')


Sample dataset created and saved to data/raw\sample_data.csv


# Homework Starter — Stage 6: Data Preprocessing
Use this notebook to apply your cleaning functions and save processed data.

In [1]:
import pandas as pd

# Uncomment once you have written src/cleaning.py (see the homework sheet).
# Until then this import fails - the module is yours to create.
# from src import cleaning

## Load Raw Dataset

In [2]:
df = pd.read_csv('data/raw/sample_data.csv')
df.head()

,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN


## Apply Cleaning Functions

In [3]:
import sys
from pathlib import Path
sys.path.append(str(Path(".").resolve()))
import src.cleaning as cleaning

df_original= df.copy()

df = cleaning.fill_missing_median(df, ['age','income','score','extra_data'])
df = cleaning.drop_missing(df, threshold=0.5)
df = cleaning.normalize_data(df, ['age','income','score','extra_data'])

## Save Cleaned Dataset

In [8]:
df.to_csv('data/processed/sample_data_cleaned.csv', index=False)

In [4]:
print("==== Shape Comparison ====")
print(f"Original dataset shape: {df_original.shape}")
print(f"Cleaned dataset shape: {df.shape}")

print("\n==== Missing Values Comparison ====")
miss_compare = pd.DataFrame({
    "original_missing": df_original.isna().sum(),
    "cleaned_missing": df.isna().sum()
})
display(miss_compare)

print("\n==== Numeric descriptive stats: Original ====")
display(df_original[['age','income','score','extra_data']].describe().T)

print("\n==== Numeric descriptive stats: Cleaned ====")
display(df[['age','income','score','extra_data']].describe().T)

==== Shape Comparison ====
Original dataset shape: (7, 6)
Cleaned dataset shape: (7, 6)

==== Missing Values Comparison ====


,original_missing,cleaned_missing
age,1,0
income,3,0
score,1,0
zipcode,0,0
city,0,0
extra_data,5,0



==== Numeric descriptive stats: Original ====


,count,mean,std,min,25%,50%,75%,max
age,6.0,39.500000,7.556454,29.00,35.0000,39.500,44.000,50.00
income,4.0,51000.000000,7071.067812,42000.00,47250.0000,52000.000,55750.000,58000.00
score,6.0,0.801667,0.092826,0.65,0.7675,0.805,0.865,0.91
extra_data,2.0,23.500000,26.162951,5.00,14.2500,23.500,32.750,42.00



==== Numeric descriptive stats: Cleaned ====


,count,mean,std,min,25%,50%,75%,max
age,7.0,0.500000,0.328479,0.0,0.333333,0.500000,0.666667,1.0
income,7.0,0.589286,0.314281,0.0,0.531250,0.625000,0.718750,1.0
score,7.0,0.585165,0.325952,0.0,0.480769,0.596154,0.769231,1.0
extra_data,7.0,0.500000,0.288675,0.0,0.500000,0.500000,0.500000,1.0


## Comparison
- The dataset kept 7 rows and 6 columns after cleaning, no rows were dropped.
- Missing values in age, income, score and extra_data were filled with column medians.
- Numeric features were scaled to [0,1] via Min‑Max normalization.

## assumption
- Median imputation: Missing values are assumed MAR. 
- 50% missing threshold: Rows with less than half valid columns would be removed.
- Min‑Max scaling: Features have valid bounds, relative distances between samples are preserved.